## XGBoost with NDVI fuel-density feature

This notebook retrains the XGBoost fire-spread model with an additional **NDVI** feature
(pre-fire vegetation density from MODIS MOD13A1, 500m, 16-day composite).

NDVI is included as a static spatial feature in the 9-cell Moore neighborhood,
expanding the feature set from 72 to 81 features.

All hyperparameters, train/test splits, and evaluation methodology are identical
to the baseline `xgb_fire_holdout.ipynb` for a fair comparison.

In [ ]:
from pathlib import Path

FIRE_SELECTION = "all"
TRAIN_FIRES = "auto"
TEST_FIRES = "auto"
FIRE_TRAIN_FRACTION = 0.70
FIRE_SPLIT_SEED = 42

POSITIVE_THRESHOLD = 0.10
CLASSIFICATION_PROB_THRESHOLD = 0.50
NORMALIZE_FEATURES = True
INCLUDE_DISCOUNTED_RAIN_FEATURE = True
INCLUDE_NDVI_FEATURE = True
DISCOUNTED_RAIN_LOOKBACK_HOURS = 24 * 30
DISCOUNTED_RAIN_HALF_LIFE_DAYS = 7.0

NEG_SUBSAMPLE_RATIO = 0.03
SUBSAMPLE_SEED = 42

N_ESTIMATORS = 500
MAX_DEPTH = 8
LEARNING_RATE = 0.05
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 0.8
MIN_CHILD_WEIGHT = 10
REG_ALPHA = 0.1
REG_LAMBDA = 1.0
TREE_METHOD = "hist"
EARLY_STOPPING_ROUNDS = 30
EVAL_FRACTION = 0.1

SEED = 1337

print("NDVI feature:", INCLUDE_NDVI_FEATURE)

In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

cwd = Path.cwd().resolve()
for candidate in [cwd] + list(cwd.parents):
    if (candidate / "scripts").exists() and (candidate / "docs").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from scripts.neighbor_cell_logreg import (
    build_feature_schema,
    collect_dataset_stats,
    discover_fire_entries,
    find_repo_root,
    fit_zscore_normalizer,
    iter_fire_hour_samples,
    safe_divide,
    select_fire_entries,
    split_fire_entries,
    split_validation_fire_entries,
)

In [ ]:
REPO_ROOT = find_repo_root(Path.cwd())
print("repo root:", REPO_ROOT)

## 1) Fire discovery and train/test split

Same split as baseline (seed=42, 70/30).

In [ ]:
all_fire_entries = discover_fire_entries(REPO_ROOT)
fire_entries = select_fire_entries(all_fire_entries, FIRE_SELECTION)
train_fire_entries, test_fire_entries = split_fire_entries(
    fire_entries, TRAIN_FIRES, TEST_FIRES, FIRE_TRAIN_FRACTION, FIRE_SPLIT_SEED,
)
print("Train fires:", [e["fire_name"] for e in train_fire_entries])
print("Test fires:", [e["fire_name"] for e in test_fire_entries])

In [ ]:
feature_schema = build_feature_schema(
    include_discounted_rain=INCLUDE_DISCOUNTED_RAIN_FEATURE,
    include_ndvi=INCLUDE_NDVI_FEATURE,
)
N_FEATURES = feature_schema.n_features
print("features per sample:", N_FEATURES)
print("vars per cell:", feature_schema.var_order)

ITERATION_KWARGS = {
    "discounted_rain_lookback_hours": DISCOUNTED_RAIN_LOOKBACK_HOURS,
    "discounted_rain_half_life_days": DISCOUNTED_RAIN_HALF_LIFE_DAYS,
}

## 2) Dataset statistics and normalization

In [ ]:
dataset_stats = collect_dataset_stats(
    train_fire_entries, test_fire_entries, REPO_ROOT,
    feature_schema, POSITIVE_THRESHOLD, **ITERATION_KWARGS,
)
ts = dataset_stats["train"]
es = dataset_stats["test"]
print(f"Train: {ts['samples']:,} samples, {ts['positives']:,} pos ({dataset_stats['train_positive_rate']:.4%})")
print(f"Test:  {es['samples']:,} samples, {es['positives']:,} pos ({dataset_stats['test_positive_rate']:.4%})")

In [ ]:
normalizer = fit_zscore_normalizer(
    train_fire_entries, REPO_ROOT, feature_schema, POSITIVE_THRESHOLD,
    enabled=NORMALIZE_FEATURES, **ITERATION_KWARGS,
)
print("normalization:", normalizer.method, "| samples:", normalizer.samples_used)

## 3) Data collection with negative subsampling

In [ ]:
def collect_subsampled_data(entries, repo_root, feature_schema, positive_threshold,
                           normalizer, neg_ratio, rng, **kwargs):
    X_pos_chunks = []
    X_neg_chunks = []
    n_pos = 0
    n_neg_total = 0
    n_neg_kept = 0

    for entry in entries:
        for _, _, X_hour, y_hour in iter_fire_hour_samples(
            entry, repo_root, feature_schema, positive_threshold, **kwargs,
        ):
            X_norm = normalizer.transform(X_hour).astype(np.float32, copy=False)
            y_bin = y_hour.astype(np.float32, copy=False)

            pos_mask = y_bin > 0.5
            neg_mask = ~pos_mask

            if pos_mask.any():
                X_pos_chunks.append(X_norm[pos_mask])
                n_pos += int(pos_mask.sum())

            if neg_mask.any():
                n_neg_hour = int(neg_mask.sum())
                n_neg_total += n_neg_hour
                keep = rng.random(n_neg_hour) < neg_ratio
                if keep.any():
                    X_neg_chunks.append(X_norm[neg_mask][keep])
                    n_neg_kept += int(keep.sum())

    X_pos = np.concatenate(X_pos_chunks, axis=0) if X_pos_chunks else np.empty((0, feature_schema.n_features), dtype=np.float32)
    X_neg = np.concatenate(X_neg_chunks, axis=0) if X_neg_chunks else np.empty((0, feature_schema.n_features), dtype=np.float32)

    X = np.concatenate([X_pos, X_neg], axis=0)
    y = np.concatenate([np.ones(n_pos, dtype=np.float32), np.zeros(n_neg_kept, dtype=np.float32)])

    perm = rng.permutation(X.shape[0])
    X = X[perm]
    y = y[perm]

    print(f"  positives: {n_pos:,}")
    print(f"  negatives: {n_neg_kept:,} / {n_neg_total:,} ({n_neg_kept/max(n_neg_total,1):.2%} kept)")
    print(f"  total: {X.shape[0]:,} samples, {X.shape[1]} features")
    print(f"  positive rate: {n_pos/max(X.shape[0],1):.4%}")
    print(f"  memory: {X.nbytes / 1e9:.2f} GB")

    return X, y, {"n_pos": n_pos, "n_neg_total": n_neg_total, "n_neg_kept": n_neg_kept}


rng = np.random.default_rng(SUBSAMPLE_SEED)

print("Collecting train data...")
X_train, y_train, train_collect_stats = collect_subsampled_data(
    train_fire_entries, REPO_ROOT, feature_schema, POSITIVE_THRESHOLD,
    normalizer, NEG_SUBSAMPLE_RATIO, rng, **ITERATION_KWARGS,
)

## 4) XGBoost training

Identical hyperparameters to baseline.

In [ ]:
n_eval = int(X_train.shape[0] * EVAL_FRACTION)
X_es, y_es = X_train[:n_eval], y_train[:n_eval]
X_tr, y_tr = X_train[n_eval:], y_train[n_eval:]

pos_count = int(y_tr.sum())
neg_count = int(y_tr.shape[0] - pos_count)
spw = neg_count / max(pos_count, 1)

print(f"Train split: {X_tr.shape[0]:,} train, {X_es.shape[0]:,} early-stop eval")
print(f"scale_pos_weight: {spw:.2f}")
print()

xgb_model = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE_BYTREE,
    min_child_weight=MIN_CHILD_WEIGHT,
    reg_alpha=REG_ALPHA,
    reg_lambda=REG_LAMBDA,
    scale_pos_weight=spw,
    tree_method=TREE_METHOD,
    eval_metric="logloss",
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    random_state=SEED,
    verbosity=1,
    n_jobs=-1,
)

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_es, y_es)],
    verbose=50,
)

best_iteration = xgb_model.best_iteration
best_score = xgb_model.best_score
print(f"\nBest iteration: {best_iteration} (logloss={best_score:.6f})")
print(f"Trees used: {best_iteration + 1} / {N_ESTIMATORS}")

## 5) Feature importance

In [ ]:
importance_gain = xgb_model.get_booster().get_score(importance_type="gain")
importance_weight = xgb_model.get_booster().get_score(importance_type="weight")

feat_names = feature_schema.feature_names
gain_vals = [(feat_names[int(k[1:])] if k.startswith("f") else k, v) for k, v in importance_gain.items()]
gain_vals.sort(key=lambda x: x[1], reverse=True)

weight_vals = [(feat_names[int(k[1:])] if k.startswith("f") else k, v) for k, v in importance_weight.items()]
weight_vals.sort(key=lambda x: x[1], reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_n = 25
names_g, vals_g = zip(*gain_vals[:top_n])
axes[0].barh(range(len(names_g)), vals_g, color="steelblue", alpha=0.8)
axes[0].set_yticks(range(len(names_g)))
axes[0].set_yticklabels(names_g, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_title(f"Top {top_n} features by gain")
axes[0].set_xlabel("Average gain")

names_w, vals_w = zip(*weight_vals[:top_n])
axes[1].barh(range(len(names_w)), vals_w, color="orangered", alpha=0.8)
axes[1].set_yticks(range(len(names_w)))
axes[1].set_yticklabels(names_w, fontsize=8)
axes[1].invert_yaxis()
axes[1].set_title(f"Top {top_n} features by split count")
axes[1].set_xlabel("Number of splits")

fig.suptitle("XGBoost feature importance (with NDVI)")
fig.tight_layout()
plt.show()

print("\nTop 15 by gain:")
for name, val in gain_vals[:15]:
    print(f"  {name:>40s}: {val:.1f}")

# Show where NDVI features rank
print("\nNDVI feature rankings by gain:")
for i, (name, val) in enumerate(gain_vals):
    if "ndvi" in name:
        print(f"  rank {i+1:>3d}: {name:>20s}: {val:.1f}")

## 6) Evaluation on held-out test fires

In [ ]:
def evaluate_xgb(model, entries, repo_root, feature_schema, positive_threshold,
                 normalizer, prob_threshold, **kwargs):
    tp = fp = fn = tn = 0
    n_eval = 0
    for entry in entries:
        for _, _, X_hour, y_hour in iter_fire_hour_samples(
            entry, repo_root, feature_schema, positive_threshold, **kwargs,
        ):
            X_eval = normalizer.transform(X_hour).astype(np.float32, copy=False)
            y_eval = y_hour.astype(np.int32, copy=False)
            probs = model.predict_proba(X_eval)[:, 1]
            y_hat = (probs >= prob_threshold).astype(np.int32)
            tp += int(((y_hat == 1) & (y_eval == 1)).sum())
            fp += int(((y_hat == 1) & (y_eval == 0)).sum())
            fn += int(((y_hat == 0) & (y_eval == 1)).sum())
            tn += int(((y_hat == 0) & (y_eval == 0)).sum())
            n_eval += int(y_eval.shape[0])
    return {
        "count": n_eval,
        "accuracy_overall": (tp + tn) / max(n_eval, 1),
        "positive_accuracy": tp / max(tp + fn, 1),
        "negative_accuracy": tn / max(tn + fp, 1),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }


metrics_test = evaluate_xgb(
    xgb_model, test_fire_entries, REPO_ROOT, feature_schema, POSITIVE_THRESHOLD,
    normalizer, CLASSIFICATION_PROB_THRESHOLD, **ITERATION_KWARGS,
)
print(f"Test samples: {metrics_test['count']:,}")
print(f"Overall accuracy: {metrics_test['accuracy_overall']:.4f}")
print(f"Positive accuracy (recall): {metrics_test['positive_accuracy']:.4f}")
print(f"Negative accuracy (specificity): {metrics_test['negative_accuracy']:.4f}")
print(f"TP={metrics_test['tp']:,}  FP={metrics_test['fp']:,}  FN={metrics_test['fn']:,}  TN={metrics_test['tn']:,}")

In [ ]:
per_fire_results = []

print("Per-fire test metrics:")
print(f"{'fire':>30s}  {'n':>10s}  {'prec':>6s}  {'rec':>6s}  {'f1':>6s}  {'TP':>7s}  {'FP':>7s}")
print("-" * 85)

for entry in test_fire_entries:
    fm = evaluate_xgb(
        xgb_model, [entry], REPO_ROOT, feature_schema, POSITIVE_THRESHOLD,
        normalizer, CLASSIFICATION_PROB_THRESHOLD, **ITERATION_KWARGS,
    )
    fm["fire_name"] = entry["fire_name"]
    prec = fm["tp"] / max(fm["tp"] + fm["fp"], 1)
    rec = fm["tp"] / max(fm["tp"] + fm["fn"], 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    fm["precision"] = prec
    fm["recall"] = rec
    fm["f1"] = f1
    per_fire_results.append(fm)
    print(f"{entry['fire_name']:>30s}  {fm['count']:>10,}  {prec:>6.4f}  {rec:>6.4f}  {f1:>6.4f}  {fm['tp']:>7,}  {fm['fp']:>7,}")

per_fire_df = pd.DataFrame(per_fire_results)
per_fire_df

## 7) Comparison with baseline (no NDVI)

Load baseline results and compare side-by-side.

In [ ]:
baseline_report_path = REPO_ROOT / "data" / "analysis" / "xgb_fire_holdout" / "report.json"
with baseline_report_path.open() as f:
    baseline_report = json.load(f)

baseline_metrics = baseline_report["metrics_test_fixed_threshold"]
baseline_per_fire = baseline_report["per_fire_test_metrics"]

# Overall comparison
print("=" * 70)
print("OVERALL TEST METRICS COMPARISON")
print("=" * 70)
print(f"{'Metric':<25s}  {'Baseline (72f)':>14s}  {'+ NDVI (81f)':>14s}  {'Delta':>10s}")
print("-" * 70)

b_acc = baseline_metrics["accuracy_overall"]
n_acc = metrics_test["accuracy_overall"]
b_rec = baseline_metrics["positive_accuracy"]
n_rec = metrics_test["positive_accuracy"]
b_spec = baseline_metrics["negative_accuracy"]
n_spec = metrics_test["negative_accuracy"]

b_prec_val = baseline_metrics["tp"] / max(baseline_metrics["tp"] + baseline_metrics["fp"], 1)
n_prec_val = metrics_test["tp"] / max(metrics_test["tp"] + metrics_test["fp"], 1)
b_f1_val = 2 * b_prec_val * b_rec / max(b_prec_val + b_rec, 1e-8)
n_f1_val = 2 * n_prec_val * n_rec / max(n_prec_val + n_rec, 1e-8)

for name, bv, nv in [
    ("Accuracy", b_acc, n_acc),
    ("Recall", b_rec, n_rec),
    ("Specificity", b_spec, n_spec),
    ("Precision", b_prec_val, n_prec_val),
    ("F1", b_f1_val, n_f1_val),
]:
    delta = nv - bv
    sign = "+" if delta >= 0 else ""
    print(f"{name:<25s}  {bv:>14.4f}  {nv:>14.4f}  {sign}{delta:>9.4f}")

print()
print(f"Baseline TP={baseline_metrics['tp']:,}  FP={baseline_metrics['fp']:,}  FN={baseline_metrics['fn']:,}  TN={baseline_metrics['tn']:,}")
print(f"NDVI     TP={metrics_test['tp']:,}  FP={metrics_test['fp']:,}  FN={metrics_test['fn']:,}  TN={metrics_test['tn']:,}")

In [ ]:
# Per-fire F1 comparison
print("\nPER-FIRE F1 COMPARISON")
print("=" * 70)
print(f"{'Fire':>30s}  {'Baseline F1':>12s}  {'NDVI F1':>12s}  {'Delta':>10s}")
print("-" * 70)

baseline_f1_map = {r["fire_name"]: r["f1"] for r in baseline_per_fire}
ndvi_f1_map = {r["fire_name"]: r["f1"] for r in per_fire_results}

deltas = []
for fire_name in sorted(baseline_f1_map.keys()):
    bf1 = baseline_f1_map.get(fire_name, 0)
    nf1 = ndvi_f1_map.get(fire_name, 0)
    d = nf1 - bf1
    deltas.append(d)
    sign = "+" if d >= 0 else ""
    print(f"{fire_name:>30s}  {bf1:>12.4f}  {nf1:>12.4f}  {sign}{d:>9.4f}")

print("-" * 70)
mean_delta = np.mean(deltas)
sign = "+" if mean_delta >= 0 else ""
print(f"{'Mean delta':>30s}  {'':>12s}  {'':>12s}  {sign}{mean_delta:>9.4f}")
print(f"\nFires improved: {sum(1 for d in deltas if d > 0)} / {len(deltas)}")
print(f"Fires degraded: {sum(1 for d in deltas if d < 0)} / {len(deltas)}")

## 8) Save report and checkpoint

In [ ]:
def to_json_safe(obj):
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Series):
        return to_json_safe(obj.to_dict())
    return obj


report = {
    "model": "xgboost_classifier_ndvi",
    "target": "center_confidence_t_plus_1_binary",
    "fires_used": [e["fire_name"] for e in fire_entries],
    "train_fires": [e["fire_name"] for e in train_fire_entries],
    "test_fires": [e["fire_name"] for e in test_fire_entries],
    "ndvi_feature": True,
    "ndvi_source": "MODIS/061/MOD13A1 (500m, 16-day, pre-fire 30-day median)",
    "thresholds": {
        "positive_confidence": POSITIVE_THRESHOLD,
        "classification_probability": CLASSIFICATION_PROB_THRESHOLD,
    },
    "architecture": {
        "type": "xgboost_gradient_boosted_trees",
        "n_estimators": N_ESTIMATORS,
        "best_iteration": best_iteration,
        "max_depth": MAX_DEPTH,
        "learning_rate": LEARNING_RATE,
        "input_features": N_FEATURES,
    },
    "metrics_test_fixed_threshold": {
        "threshold": CLASSIFICATION_PROB_THRESHOLD,
        **metrics_test,
    },
    "per_fire_test_metrics": per_fire_results,
    "comparison_vs_baseline": {
        "baseline_accuracy": b_acc,
        "ndvi_accuracy": n_acc,
        "baseline_recall": b_rec,
        "ndvi_recall": n_rec,
        "baseline_f1": b_f1_val,
        "ndvi_f1": n_f1_val,
        "accuracy_delta": n_acc - b_acc,
        "recall_delta": n_rec - b_rec,
        "f1_delta": n_f1_val - b_f1_val,
    },
    "feature_importance_gain_top15": gain_vals[:15],
    "feature_order": feature_schema.feature_names,
}

report = to_json_safe(report)
out_dir = REPO_ROOT / "data" / "analysis" / "xgb_fire_holdout_ndvi"
out_dir.mkdir(parents=True, exist_ok=True)
report_path = out_dir / "report.json"
with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("saved:", report_path)

# Save checkpoint
ckpt_dir = REPO_ROOT / "data" / "checkpoints"
xgb_model.save_model(str(ckpt_dir / "xgboost_ndvi.json"))

meta = {
    "normalizer_mean": normalizer.mean.tolist(),
    "normalizer_std": normalizer.std_safe.tolist(),
    "include_discounted_rain": INCLUDE_DISCOUNTED_RAIN_FEATURE,
    "include_ndvi": INCLUDE_NDVI_FEATURE,
    "feature_names": feature_schema.feature_names,
    "train_fires": [e["fire_name"] for e in train_fire_entries],
    "test_fires": [e["fire_name"] for e in test_fire_entries],
}
with open(ckpt_dir / "xgboost_ndvi_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved checkpoint to {ckpt_dir / 'xgboost_ndvi.json'}")
print(f"Saved metadata to {ckpt_dir / 'xgboost_ndvi_meta.json'}")